# Phase 8a — Local Dev Smoke Test

Verifies the local dev setup per Phase 8a:
1. The `gemini_hackathon` Python package imports cleanly
2. The ADK backend module loads
3. The observability module's `init_backend_observability()` runs
   without crashing (env-gated; no Langfuse/MLflow required for the
   in-process smoke test)
4. The 3-tier model policy is correctly configured
5. The BAML clients regenerate cleanly

If the Docker Compose stack is running (docker compose -f
docker-compose.yml -f docker-compose.local.yaml up), the env vars
LANGFUSE_PUBLIC_KEY + MLFLOW_TRACKING_URI get auto-detected and the
observability state will show them as active.

In [ ]:
# 1. Core imports.
import importlib

# gemini_hackathon package
import gemini_hackathon
print(f"gemini_hackathon ok (callable: {callable(gemini_hackathon)})")

# ADK backend
import gemini_hackathon_backend  # noqa: F401
print("gemini_hackathon_backend ok")

# Observability
from gemini_hackathon_backend import observability
state = observability.init_backend_observability()
print(f"observability state: {state}")

In [ ]:
# 2. 3-tier model policy.
from gemini_hackathon.call_llm import (
    HACKATHON_TIERS,
    build_model_list,
)
print(f"HACKATHON_TIERS: {HACKATHON_TIERS}")
model_list, fallbacks = build_model_list()
for m in model_list:
    print(f"  {m['model_name']:8s} -> {m['litellm_params']['model']}")
print(f"fallbacks: {fallbacks}")

In [ ]:
# 3. Memory service factory.
from gemini_hackathon_backend.agents.memory import (
    build_memory_service,
    memory_user_id,
    memory_root,
)
mem = build_memory_service()
print(f"memory_service: {mem}  (None = fallback to InMemoryMemoryService)")
print(f"memory_user_id: {memory_user_id()}")
root = memory_root()
print(f"memory_root: {root}  (None = markdown path not activated)")

In [ ]:
# 4. Pipeline modules.
import importlib.util
_load = lambda name, path: importlib.util.spec_from_file_location(
    name, path
).loader.exec_module(importlib.util.module_from_spec(importlib.util.spec_from_file_location(name, path)))

import sys as _sys
def _load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    _sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

import pathlib
_base = pathlib.Path(".")
_pd = _load_module("_test_pd", _base / "dlt_pipelines" / "pdf_downloader.py")
_md = _load_module("_test_md", _base / "cocoindex_flows" / "pdf" / "pdf_to_markdown_app.py")
_baml = _load_module("_test_baml", _base / "cocoindex_flows" / "education" / "lc6_extraction_app.py")
_eq = _load_module("_test_eq", _base / "cocoindex_flows" / "equivalency" / "equivalency_graph_app.py")

print(f"dlt_pipelines.pdf_downloader:        ok (PDF_RAW_ROOT={_pd.PDF_RAW_ROOT.name})")
print(f"cocoindex_flows.pdf.pdf_to_markdown:  ok (MD_ROOT={_md.MD_ROOT.name})")
print(f"cocoindex_flows.education.lc6:        ok (SQLITE_PATH={_baml.SQLITE_PATH.name})")
print(f"cocoindex_flows.equivalency.equiv:     ok (SQLITE_PATH={_eq.SQLITE_PATH.name})")

## Summary

- Local dev surface: `docker compose -f docker-compose.yml -f docker-compose.local.yaml up` boots Langfuse (`:3000`) + MLflow (`:5000`) + the application + llama-swap.
- Env-gated: with no Langfuse key + no MLflow URI, all observability functions no-op cleanly (verified in cell 1).
- The 3-tier model policy (MiniMax-M3 → Unsloth → Vertex Agent Garden) is wired and the router config is correct (cell 2).
- The memory service falls through to InMemoryMemoryService when neither DEPLOYED_AGENT_ENGINE_ID nor GH_MEMORY_DIR is set (cell 3).
- All 4 CocoIndex pipeline modules import cleanly (cell 4).
- Ready for Phase 8c (observability verification integration test) + the dev Cloud Run deploy (8b).